# E1 — Warm-start RL: does RL help *after* CE?

The paper shows cold-start RL fails at 0.5B. Here we start RL from the
**CE-trained** policy (~0.98) and ask whether RL refinement adds points,
holds steady, or degrades it. Three arms, from the released CE checkpoint:

- **W1 warm+REINFORCE** — sampled policy gradient from the warm policy.
- **W2 warm+KL-REINFORCE** — W1 + KL penalty toward the frozen warm policy
  (coef swept on seed 7, then 3 seeds at the best).
- **W3 warm+exact-PG** — closed-form expected-reward gradient from warm.

Every final checkpoint is evaluated on the committed test AND a fresh set,
with Wilson CIs; the reported quantity is the delta vs. the CE start.

Runtime → T4/L4 GPU → Run all. Download `warmstart_results.zip` at the end.

In [ ]:
import os
os.chdir('/content')
if not os.path.isdir('verifier-as-reward'):
    !git clone https://github.com/esmaeil-abedi-dev/verifier-as-reward.git
os.chdir('/content/verifier-as-reward')
print('cwd:', os.getcwd())
import torch; assert torch.cuda.is_available()
WARM = 'esmaeil-abedi-dev/verifier-ce-qwen2.5-0.5b'  # the CE warm-start policy
!PYTHONPATH=. python test_authority_verifier.py
!PYTHONPATH=. python make_expanded_train.py --train-seed 101 \
  --train-traces-per-class 150 --val-seed 202 --val-traces-per-class 25

## Step 1 — KL-coefficient sweep on seed 7 (pick the best on VAL)

In [ ]:
# W2 KL-coefficient search on seed 7 (monitored on val). Python os.system
# so WARM interpolates reliably (bash !for + $WARM does not).
import os, json
for kl in ['0.02', '0.1', '0.5']:
    print(f'=== KL={kl} ===')
    rc = os.system(
        'PYTHONPATH=. python train_verifier_reward.py '
        '--model Qwen/Qwen2.5-0.5B '
        f'--warm-start-from {WARM} --kl-coef {kl} '
        '--balance-reward --prompt-style nl '
        '--steps 500 --batch-size 16 --lr 2e-5 '
        '--train-file expanded_train.jsonl --test-file expanded_val.jsonl '
        '--eval-every 25 --eval-max-actions 120 --seed 7 '
        f'--log-file training_log_warmstart_klsweep_{kl}.jsonl')
    assert rc == 0, f'KL={kl} run failed (rc={rc})'
for kl in ['0.02', '0.1', '0.5']:
    pts = [json.loads(l) for l in open(f'training_log_warmstart_klsweep_{kl}.jsonl')
           if '\"step\"' in l]
    e = pts[-1]
    print(f"KL={kl}: final val acc {e['accuracy']:.3f} "
          f"viol {e['heldout_violation_rate']} fref {e['heldout_false_refuse_rate']}")


In [ ]:
BEST_KL = '0.1'  # <-- set from the sweep above (highest val acc, both errors low)
print('using KL =', BEST_KL)

## Step 2 — three arms x three seeds (train)

In [ ]:
# W1 warm+REINFORCE, W2 warm+KL (best coef), W3 warm+exact-PG; seeds 7,8,9.
import os
for arm, extra in [('w1_reinforce', '--balance-reward'),
                   ('w2_kl', f'--balance-reward --kl-coef {BEST_KL}'),
                   ('w3_exactpg', '--exact-pg --balance-reward')]:
    for s in [7, 8, 9]:
        print(f'=== {arm} seed {s} ===')
        os.system(f'PYTHONPATH=. python train_verifier_reward.py '
                  f'--model Qwen/Qwen2.5-0.5B --warm-start-from {WARM} '
                  f'--prompt-style nl --steps 500 --batch-size 16 --lr 2e-5 '
                  f'--train-file expanded_train.jsonl --test-file expanded_val.jsonl '
                  f'--eval-every 25 --eval-max-actions 120 --seed {s} {extra} '
                  f'--log-file training_log_warmstart_{arm}_seed{s}.jsonl '
                  f'--save-dir ckpt_warmstart_{arm}_seed{s}')

## Step 3 — evaluate every arm on committed test + fresh set (vs the CE start)

In [ ]:
# Fresh in-distribution set (new seed, deduped vs train) for a tighter CI.
from trace_benchmark import generate_corpus, write_jsonl, load_traces
from make_expanded_train import corpus_canonicals, drop_overlapping
test_canon = corpus_canonicals(load_traces('benchmark_test.jsonl'))
a,b = generate_corpus(101,150); train_canon = corpus_canonicals(drop_overlapping(a+b,test_canon,'train'))
c,d = generate_corpus(770,60)
fresh = drop_overlapping(c+d, train_canon|test_canon, 'fresh')
write_jsonl(fresh, 'warmstart_fresh.jsonl'); print('fresh actions:', sum(len(t['actions']) for t in fresh))

In [ ]:
import os, json
json.dump({'backends': {}}, open('results_warmstart.json', 'w'))
def score(ckpt, tf):
    rc = os.system(f'PYTHONPATH=. python train_verifier_reward.py '
                   f'--eval-checkpoint {ckpt} --test-file {tf} '
                   f'--merge-results results_warmstart.json')
    assert rc == 0, f'eval failed: {ckpt} on {tf}'
# baseline row: the CE start itself, on both sets
for tf in ['benchmark_test.jsonl', 'warmstart_fresh.jsonl']:
    score(WARM, tf)
for arm in ['w1_reinforce', 'w2_kl', 'w3_exactpg']:
    for s in [7, 8, 9]:
        ck = f'ckpt_warmstart_{arm}_seed{s}'
        if not os.path.isdir(ck):
            continue
        for tf in ['benchmark_test.jsonl', 'warmstart_fresh.jsonl']:
            score(ck, tf)
print('done ->', json.load(open('results_warmstart.json'))['backends'].keys())


In [ ]:
!zip -j warmstart_results.zip training_log_warmstart_*.jsonl results_warmstart.json
from google.colab import files
files.download('warmstart_results.zip')